In [2]:

import sys

import pandas as pd
from fowt_ml import Config
from fowt_ml.datasets import get_data_mfiles
from fowt_ml.pipeline import Pipeline
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

sys.path.append("../")
from utils import DATA_DECAY_CL_TEST, RECORDINGS_CSV

In [ ]:

# constants

DATA_FOLDER = DATA_DECAY_CL_TEST
KHZ_SCALE = 1e3  # scale factor; kHz to seconds

## 1 Model selection

### Create a config

In [3]:
# load the config
config_file = "./experiment_force_correction.yml"
config = Config.from_yaml(config_file)

ml_setup = config["ml_setup"]

In [4]:
# set the model names; we selected three models
# TODO: hyperparameters are as of yet untouched
# TODO: can do some tuning within the pipeline/on the validation set?
ml_setup["model_names"] = {
    "LinearRegression": {},
    "MultilayerPerceptron": {
        "hidden_layer_sizes": (8, 8, 8, 8, 8, 8),
        "activation": "relu",
        "max_iter": 500,
        "alpha": 0.00001,
        "learning_rate": "adaptive",
        "validation_fraction": 0.15,
        "early_stopping": True,
        "n_iter_no_change": 15,
        "random_state": 42,
    },
    "XGBoostRegression": {
        "n_estimators": 500,
        "max_depth": 10,
        "learning_rate": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "colsample_bylevel": 0.7,
        "min_child_weight": 5,
        "gamma": 0.0,
        "tree_method": "hist",
        "random_state": 42,
    },
}

{}

In [5]:
# set shuffle to True because it is instantaneous prediction
ml_setup["train_test_split_kwargs"] = {
    "train_size": 0.80,
    "shuffle": True,
    "random_state": 42,
}

In [ ]:

# closed-lopp, surge, sway, heave, roll, pitch, yaw, no wind, no-rotation
data_ids = ["exp1644", "exp1646", "exp1648", "exp1651", "exp1653", "exp1656"]

### Compare models with different data

In [9]:
# get all data
! this opens all exp in the folder and brings them into one dataframe
! added the aux recording data, so each row knows its experiment
df = get_data_mfiles(
    DATA_FOLDER, aux_csv_file=RECORDINGS_CSV, aux_df_column_name="recording_number"
)

# TODO: is this useful?! Last data gets to e.g. 2052 days, which misrepresents the time scales of the concat'd data (not one timeseries)
# TODO: needs to be time per experiment, not across experiments
df.index = pd.to_timedelta(df.index * KHZ_SCALE, unit="s")  # set time index

! grouped data; and data is only predictors and targets, no aux data
! this is because setup procedure currently donesn't handle non-numeric data in aux; see vault notes
# TODO: this subsetting shouldn't be needed when addressed upstream
ml_setup = config["ml_setup"]
groups = df["recording_number"]  # for grouped splitting / per-experiment scoring
df_ml = df[ml_setup["predictors"] + ml_setup["targets"]]  # all numeric, no NaNs

# create holdout data (test) with shuffle
# TODO: change to without shuffle no?!
data_trainval, data_test = train_test_split(
    df_ml, test_size=0.2, shuffle=True, random_state=42
)

# create pipline and set it up
comparison_pipeline = Pipeline(config)
comparison_pipeline.setup(data=data_trainval)

# compare models
# TODO: should use cross validation?
# TODO: there's a TODO in the FOWT-ML.compare_models() to parallise the comparison, which would be useful here...
models, scores = comparison_pipeline.compare_models(sort="r2")

# validation on holdout data
best_model = scores.index[0]
model = models[best_model]
y_predict = model.predict(data_test[comparison_pipeline.predictors_labels].values)
y_target = data_test[comparison_pipeline.target_labels].values
r2 = r2_score(y_target, y_predict, multioutput="raw_values")

train size is 141844 and validation size is 35462.
Scores of the models are:
                      neg_mean_absolute_error        r2
XGBoostRegression                   -0.105725  0.660569
MultilayerPerceptron                -0.105815  0.572488
LinearRegression                    -0.108272  0.565529
r2 of best model on validation set are: [0.96838168 0.94677038 0.00777549 0.86711239 0.76231532 0.44804483]


### Save results for the dashboard

In [ ]:
# save what `ml_dashboard.ipynb` needs, so the results can be explored there without re-training
# N.B. here, `inventory/` is a local working area (excluded from git) and will be user specific
! this is a tmp solution, workflows upstream in fowt-ml could support this (see https://github.com/hybridlabs-nl/FOWT-ML/issues/50)
import json
from datetime import datetime
from pathlib import Path

import joblib

RESULTS_DIR = Path("../inventory/ml_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# hold-out rows, labelled with their experiment and per-experiment time
test_data = data_test.join(df[["recording_number", "time"]])
test_data.reset_index(drop=True).to_parquet(RESULTS_DIR / "test_data.parquet")

# fitted models and the pipeline's own (averaged) scores
joblib.dump(models, RESULTS_DIR / "models.joblib")
scores.to_csv(RESULTS_DIR / "scores.csv", index_label="model")

# run settings, so the dashboard can say what it's showing
run_info = {
    "created": datetime.now().isoformat(timespec="seconds"),
    "data_folder": str(DATA_FOLDER),
    "recordings": sorted(int(r) for r in df["recording_number"].unique()),
    "n_rows": len(df_ml),
    "n_trainval": len(data_trainval),
    "n_test": len(data_test),
    "holdout_split": "train_test_split(test_size=0.2, shuffle=True, random_state=42)",
    "pipeline_split": ml_setup["train_test_split_kwargs"],
    "predictors": ml_setup["predictors"],
    "targets": ml_setup["targets"],
    "metric_names": ml_setup["metric_names"],
    "model_names": ml_setup["model_names"],
    "best_model": best_model,
}
with open(RESULTS_DIR / "run_info.json", "w") as f:
    json.dump(run_info, f, indent=2, default=str)
print(f"Saved results to {RESULTS_DIR.resolve()}")